In [1]:
%pip install pandera

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import pandera
import pandera.pandas as pa

print("pandas :", pd.__version__)
print("pandera :", pandera.__version__)

erp = pd.read_excel('erp.xlsx')
web = pd.read_excel('web.xlsx')
liaison = pd.read_excel('liaison.xlsx')
print(erp.shape, web.shape, liaison.shape)

pandas : 3.0.2
pandera : 0.34.1
(825, 6) (1513, 29) (825, 2)


/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/openpyxl/worksheet/_read_only.py:85: UserWarning: Unknown extension is not supported and will be removed
  for idx, row in parser.parse():
/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/openpyxl/worksheet/_read_only.py:85: UserWarning: Unknown extension is not supported and will be removed
  for idx, row in parser.parse():
/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/openpyxl/worksheet/_read_only.py:85: UserWarning: Unknown extension is not supported and will be removed
  for idx, row in parser.parse():


In [3]:
# Gravité de chaque règle, reprise du catalogue (tâche 2)
GRAVITE = {"R1": "Bloquante", "R2": "Bloquante", "R3": "Avertissement", "R4": "Bloquante"}

schema_erp = pa.DataFrameSchema(
    columns={
        "product_id": pa.Column(int, unique=True, nullable=False),
        "onsale_web": pa.Column(int, pa.Check.isin([0, 1]), nullable=False),
        "price": pa.Column(float, pa.Check.gt(0, name="R1",
                  error="R1 – Le prix de vente doit être strictement positif"), nullable=False),
        "stock_quantity": pa.Column(int, pa.Check.ge(0, name="R2",
                  error="R2 – La quantité en stock ne doit jamais être négative"), nullable=False),
        "stock_status": pa.Column(checks=pa.Check.isin(["instock", "outofstock"]), nullable=False),
        "purchase_price": pa.Column(float, nullable=False),
    },
    checks=[
        # R3 : en stock si quantité > 0, en rupture sinon
        pa.Check(lambda df: ((df["stock_quantity"] > 0) & (df["stock_status"] == "instock"))
                          | ((df["stock_quantity"] <= 0) & (df["stock_status"] == "outofstock")),
                 name="R3", error="R3 – Le statut de stock ne correspond pas à la quantité"),
        # R4 : prix de vente >= prix d'achat (les prix négatifs relèvent déjà de R1)
        pa.Check(lambda df: (df["price"] <= 0) | (df["price"] >= df["purchase_price"]),
                 name="R4", error="R4 – Le prix de vente est inférieur au prix d'achat"),
    ],
)

In [4]:
try:
    schema_erp.validate(erp, lazy=True)
    print("ERP : aucune anomalie")
except pa.errors.SchemaErrors as e:
    erreurs_erp = e.failure_cases
    print("Nombre de lignes en anomalie par règle :")
    display(erreurs_erp.groupby("check")["index"].nunique())
    display(erreurs_erp)

Nombre de lignes en anomalie par règle :


check
R1 – Le prix de vente doit être strictement positif        3
R2 – La quantité en stock ne doit jamais être négative     2
R3 – Le statut de stock ne correspond pas à la quantité    2
R4 – Le prix de vente est inférieur au prix d'achat        4
Name: index, dtype: int64

,schema_context,column,check,check_number,failure_case,index
5,DataFrameSchema,product_id,R3 – Le statut de stock ne correspond pas à la...,0,4039,4
6,DataFrameSchema,product_id,R3 – Le statut de stock ne correspond pas à la...,0,4885,398
7,DataFrameSchema,onsale_web,R3 – Le statut de stock ne correspond pas à la...,0,1,4
8,DataFrameSchema,onsale_web,R3 – Le statut de stock ne correspond pas à la...,0,1,398
9,DataFrameSchema,price,R3 – Le statut de stock ne correspond pas à la...,0,46.0,4
10,DataFrameSchema,price,R3 – Le statut de stock ne correspond pas à la...,0,18.7,398
11,DataFrameSchema,stock_quantity,R3 – Le statut de stock ne correspond pas à la...,0,3,4
12,DataFrameSchema,stock_quantity,R3 – Le statut de stock ne correspond pas à la...,0,0,398
13,DataFrameSchema,stock_status,R3 – Le statut de stock ne correspond pas à la...,0,outofstock,4
14,DataFrameSchema,stock_status,R3 – Le statut de stock ne correspond pas à la...,0,instock,398


In [5]:
GRAVITE["R5"] = "Bloquante"

schema_liaison = pa.DataFrameSchema(
    columns={
        "product_id": pa.Column(int, unique=True, nullable=False),
        # id_web peut être vide (produits vendus uniquement en magasin)
        "id_web": pa.Column(
            checks=pa.Check(lambda s: s.astype(str).str.fullmatch(r"\d+"), name="R5",
                            error="R5 – L'identifiant web doit être numérique"),
            nullable=True,
        ),
    },
)

try:
    schema_liaison.validate(liaison, lazy=True)
    print("Liaison : aucune anomalie")
except pa.errors.SchemaErrors as e:
    erreurs_liaison = e.failure_cases
    print("Nombre de lignes en anomalie par règle :")
    display(erreurs_liaison.groupby("check")["index"].nunique())
    display(erreurs_liaison)

Nombre de lignes en anomalie par règle :


check
R5 – L'identifiant web doit être numérique    3
Name: index, dtype: int64

,schema_context,column,check,check_number,failure_case,index
0,Column,id_web,R5 – L'identifiant web doit être numérique,0,bon-cadeau-25-euros,443
1,Column,id_web,R5 – L'identifiant web doit être numérique,0,13127-1,822
2,Column,id_web,R5 – L'identifiant web doit être numérique,0,14680-1,823


In [6]:
GRAVITE.update({"R8": "Bloquante", "R9": "Bloquante", "R11": "Avertissement", "R14": "Avertissement"})

MOTIF_HTML = r"<[^>]+>|&[a-zA-Z]+;|&#\d+;"   # balises (<div>) et entités (&amp;)

def est_produit(df):
    return df["post_type"] == "product"

def sku_en_doublon(df):
    produits = est_produit(df) & df["sku"].notna()
    doublons = df.loc[produits, "sku"].astype(str).duplicated(keep=False)
    return doublons.reindex(df.index, fill_value=False)

schema_web = pa.DataFrameSchema(
    checks=[
        pa.Check(lambda df: est_produit(df), name="R8",
                 error="R8 – La ligne n'est pas un produit (image, ligne vide…)"),
        pa.Check(lambda df: ~sku_en_doublon(df), name="R8",
                 error="R8 – Le produit apparaît plusieurs fois dans l'export"),
        pa.Check(lambda df: ~est_produit(df) | df["sku"].isna()
                            | df["sku"].astype(str).str.fullmatch(r"\d+"), name="R5",
                 error="R5 – La référence web doit être numérique"),
        pa.Check(lambda df: ~est_produit(df) | (df["sku"].notna() & (df["total_sales"] >= 0)), name="R9",
                 error="R9 – Le produit doit avoir une référence et des ventes positives ou nulles"),
        pa.Check(lambda df: ~est_produit(df)
                            | ~(df["post_title"].astype(str).str.contains(MOTIF_HTML)
                                | df["post_excerpt"].astype(str).str.contains(MOTIF_HTML)), name="R11",
                 error="R11 – Le titre ou la description contient du code HTML"),
        pa.Check(lambda df: ~est_produit(df) | df["product_type"].notna(), name="R14",
                 error="R14 – Le produit n'a pas de type (ex. bon cadeau)"),
    ],
)

try:
    schema_web.validate(web, lazy=True)
    print("Web : aucune anomalie")
except pa.errors.SchemaErrors as e:
    erreurs_web = e.failure_cases
    print("Nombre de lignes en anomalie par règle :")
    display(erreurs_web.groupby("check")["index"].nunique())

Nombre de lignes en anomalie par règle :


check
R11 – Le titre ou la description contient du code HTML                        183
R14 – Le produit n'a pas de type (ex. bon cadeau)                               1
R5 – La référence web doit être numérique                                       2
R8 – La ligne n'est pas un produit (image, ligne vide…)                       797
R9 – Le produit doit avoir une référence et des ventes positives ou nulles      2
Name: index, dtype: int64

In [7]:
produits = web[web["post_type"] == "product"]
desc = produits["post_excerpt"].astype(str)
titre = produits["post_title"].astype(str)
ENTITES = r"&[a-zA-Z]+;|&#\d+;"

balises = desc.str.contains(r"<[^>]+>")
entites_desc = desc.str.contains(ENTITES)

print("Descriptions avec balises :", balises.sum())
print("Descriptions avec entités mais sans balise :", (entites_desc & ~balises).sum())
print("Titres avec entités :", titre.str.contains(ENTITES).sum())
print("\nEntités les plus fréquentes dans les descriptions :")
display(desc.str.findall(ENTITES).explode().value_counts().head(10))

Descriptions avec balises : 146
Descriptions avec entités mais sans balise : 32
Titres avec entités : 5

Entités les plus fréquentes dans les descriptions :


post_excerpt
&nbsp;    65
Name: count, dtype: int64

In [8]:
GRAVITE["R10"] = "Avertissement"

nb_valeurs = web.nunique(dropna=True)

colonnes_vides = nb_valeurs[nb_valeurs == 0].index
colonnes_constantes = nb_valeurs[nb_valeurs == 1].index

# Une ligne par colonne signalée, avec le message adapté à chaque cas
anomalies_r10 = pd.DataFrame(
    [{"Règle": "R10", "Table": "web", "Colonne": col,
      "Message": "R10 – Colonne vide : à retirer de l'export"} for col in colonnes_vides]
  + [{"Règle": "R10", "Table": "web", "Colonne": col,
      "Message": "R10 – Colonne constante : utilité à confirmer avec Nicolas"} for col in colonnes_constantes]
)

print("Colonnes vides :", len(colonnes_vides))
print("Colonnes constantes :", len(colonnes_constantes))
display(anomalies_r10)

Colonnes vides : 4
Colonnes constantes : 12


,Règle,Table,Colonne,Message
0,R10,web,tax_class,R10 – Colonne vide : à retirer de l'export
1,R10,web,post_content,R10 – Colonne vide : à retirer de l'export
2,R10,web,post_password,R10 – Colonne vide : à retirer de l'export
3,R10,web,post_content_filtered,R10 – Colonne vide : à retirer de l'export
4,R10,web,virtual,R10 – Colonne constante : utilité à confirmer ...
5,R10,web,downloadable,R10 – Colonne constante : utilité à confirmer ...
6,R10,web,rating_count,R10 – Colonne constante : utilité à confirmer ...
7,R10,web,average_rating,R10 – Colonne constante : utilité à confirmer ...
8,R10,web,tax_status,R10 – Colonne constante : utilité à confirmer ...
9,R10,web,post_status,R10 – Colonne constante : utilité à confirmer ...


In [9]:
GRAVITE.update({"R6": "Avertissement", "R7": "Avertissement", "R12": "Avertissement", "R13": "Bloquante"})

# Identifiants comparés en texte pour neutraliser le mélange nombres / texte (R5)
liaison_txt = liaison.assign(id_web=liaison["id_web"].astype("string"))
produits_web = web[web["post_type"] == "product"].dropna(subset=["sku"])
skus_web = set(produits_web["sku"].astype(str))
ids_liaison = set(liaison_txt["id_web"].dropna())

erp_liaison = erp.merge(liaison_txt, on="product_id", how="left")

# R6 : en vente web mais sans identifiant web
r6 = erp_liaison[(erp_liaison["onsale_web"] == 1) & erp_liaison["id_web"].isna()]
# R7 : retiré de la vente web mais avec un identifiant web
r7 = erp_liaison[(erp_liaison["onsale_web"] == 0) & erp_liaison["id_web"].notna()]
# R12 : référence de la liaison absente du site
r12 = liaison_txt[liaison_txt["id_web"].notna() & ~liaison_txt["id_web"].isin(skus_web)]
# R13 : produit du site sans correspondance dans la liaison, donc dans l'ERP
r13 = produits_web[~produits_web["sku"].astype(str).isin(ids_liaison)]

for regle, df in [("R6", r6), ("R7", r7), ("R12", r12), ("R13", r13)]:
    print(regle, ":", len(df), "ligne(s)")

R6 : 3 ligne(s)
R7 : 21 ligne(s)
R12 : 20 ligne(s)
R13 : 0 ligne(s)


In [10]:
# À quel produit ERP correspond l'identifiant web 14736 ?
correspondance = liaison_txt[liaison_txt["id_web"] == "14736"]
display(correspondance)
display(erp[erp["product_id"].isin(correspondance["product_id"])])
display(produits_web[produits_web["sku"].astype(str) == "14736"][["sku", "post_title", "total_sales"]])

,id_web,product_id
573,14736,5700


,product_id,onsale_web,price,stock_quantity,stock_status,purchase_price
573,5700,1,44.5,-1,outofstock,22.3


,sku,post_title,total_sales
1126,14736,"Gilles Robin Crozes-Hermitage Rouge ""1920"" 2016",8.0


In [11]:
GRAVITE["R15"] = "Avertissement"

# Titres des produits du site, avec la référence en texte
titres = produits_web.assign(sku=produits_web["sku"].astype(str))[["sku", "post_title"]]

# Pour chaque produit ERP : liaison, puis titre du site
erp_titres = erp_liaison.merge(titres, left_on="id_web", right_on="sku", how="left")

# R15 : aucun libellé disponible, ni dans l'ERP ni via le site
r15 = erp_titres[erp_titres["post_title"].isna()]

print("R15 :", len(r15), "produit(s) sans libellé")
display(r15[["product_id", "onsale_web", "id_web"]].head(10))

R15 : 111 produit(s) sans libellé


,product_id,onsale_web,id_web
19,4055,0,<NA>
49,4090,0,<NA>
50,4092,0,<NA>
119,4195,0,<NA>
131,4209,0,<NA>
151,4233,0,<NA>
184,4278,0,<NA>
185,4279,0,<NA>
193,4289,0,13771
234,4565,0,<NA>


In [12]:
import re

GRAVITE["Structure"] = "Bloquante"

def lignes_pandera(schema, df, table, colonne_id):
    """Valide un schéma Pandera et renvoie une ligne par anomalie et par élément concerné."""
    try:
        schema.validate(df, lazy=True)
        return []
    except pa.errors.SchemaErrors as e:
        cas = e.failure_cases
    # Une seule ligne par couple (contrôle, ligne du tableau), quel que soit le nombre de colonnes
    cas = cas.drop_duplicates(subset=["check", "index"])
    lignes = []
    for _, c in cas.iterrows():
        message = str(c["check"])
        trouve = re.match(r"(R\d+)", message)
        regle = trouve.group(1) if trouve else "Structure"
        if regle == "Structure":
            message = f"Structure – {c['column']} : contrôle « {message} » non respecté ({c['failure_case']})"
        if pd.notna(c["index"]):
            ident = df.loc[c["index"], colonne_id]
            ident = c["index"] if pd.isna(ident) else ident
        else:
            ident = c["column"]
        lignes.append({"Règle": regle, "Table": table, "Identifiant": str(ident), "Message": message})
    return lignes


def controler(erp, web, liaison):
    """Applique les 15 règles et renvoie un tableau unique d'anomalies, sans modifier les données."""
    lignes = []
    lignes += lignes_pandera(schema_erp, erp, "ERP", "product_id")
    lignes += lignes_pandera(schema_liaison, liaison, "Liaison", "product_id")
    lignes += lignes_pandera(schema_web, web, "Web", "sku")

    # R10 : colonnes vides ou constantes
    nb = web.nunique(dropna=True)
    for col in nb[nb == 0].index:
        lignes.append({"Règle": "R10", "Table": "Web", "Identifiant": col,
                       "Message": "R10 – Colonne vide : à retirer de l'export"})
    for col in nb[nb == 1].index:
        lignes.append({"Règle": "R10", "Table": "Web", "Identifiant": col,
                       "Message": "R10 – Colonne constante : utilité à confirmer avec Nicolas"})

    # Règles inter-tables, identifiants comparés en texte
    liaison_txt = liaison.assign(id_web=liaison["id_web"].astype("string"))
    produits_web = web[web["post_type"] == "product"].dropna(subset=["sku"])
    produits_web = produits_web.assign(sku=produits_web["sku"].astype(str))
    skus_web = set(produits_web["sku"])
    ids_liaison = set(liaison_txt["id_web"].dropna())
    erp_liaison = erp.merge(liaison_txt, on="product_id", how="left")
    erp_titres = erp_liaison.merge(produits_web[["sku", "post_title"]],
                                   left_on="id_web", right_on="sku", how="left")

    controles = [
        ("R6", "ERP / liaison", erp_liaison[(erp_liaison["onsale_web"] == 1) & erp_liaison["id_web"].isna()],
         "product_id", "R6 – Produit en vente web sans identifiant web"),
        ("R7", "ERP / liaison", erp_liaison[(erp_liaison["onsale_web"] == 0) & erp_liaison["id_web"].notna()],
         "product_id", "R7 – Produit retiré de la vente web avec un identifiant web"),
        ("R12", "Liaison / web", liaison_txt[liaison_txt["id_web"].notna() & ~liaison_txt["id_web"].isin(skus_web)],
         "product_id", "R12 – Référence de la liaison absente du site"),
        ("R13", "Web / liaison", produits_web[~produits_web["sku"].isin(ids_liaison)],
         "sku", "R13 – Produit du site absent de la liaison et de l'ERP"),
        ("R15", "ERP", erp_titres[erp_titres["post_title"].isna()],
         "product_id", "R15 – Aucun libellé disponible, ni dans l'ERP ni sur le site"),
    ]
    for regle, table, df, col, message in controles:
        for ident in df[col]:
            lignes.append({"Règle": regle, "Table": table, "Identifiant": str(ident), "Message": message})

    rapport = pd.DataFrame(lignes)
    rapport.insert(1, "Gravité", rapport["Règle"].map(GRAVITE))
    return rapport

In [13]:
# 1. Contrôle de l'export d'octobre (référence) et de l'export dégradé
octobre = controler(erp, web, liaison)
degrade = controler(pd.read_excel('erp_degrade.xlsx'),
                    pd.read_excel('web_degrade.xlsx'),
                    pd.read_excel('liaison_degrade.xlsx'))
print("Anomalies octobre :", len(octobre), "| export dégradé :", len(degrade))

# 2. Anomalies nouvelles, absentes de l'export d'octobre
cle = ["Règle", "Identifiant", "Message"]
nouvelles = (degrade.merge(octobre[cle], on=cle, how="left", indicator=True)
                    .query("_merge == 'left_only'").drop(columns="_merge"))
print("Nouvelles anomalies :", len(nouvelles))
display(nouvelles)

# 3. Chaque erreur injectée est-elle détectée par sa règle ?
injections = pd.read_csv('injections.csv', dtype=str)
injections["Détectée"] = [((nouvelles["Règle"] == r["Règle"]) & (nouvelles["Identifiant"] == r["Identifiant"])).any()
                          for _, r in injections.iterrows()]
display(injections)
print(f"Taux de détection : {injections['Détectée'].mean():.0%}")

Anomalies octobre : 1170 | export dégradé : 1192
Nouvelles anomalies : 22


,Règle,Gravité,Table,Identifiant,Message
1,R3,Avertissement,ERP,4275,R3 – Le statut de stock ne correspond pas à la...
2,R3,Avertissement,ERP,4713,R3 – Le statut de stock ne correspond pas à la...
4,R3,Avertissement,ERP,6585,R3 – Le statut de stock ne correspond pas à la...
6,R4,Bloquante,ERP,4632,R4 – Le prix de vente est inférieur au prix d'...
12,R1,Bloquante,ERP,6221,R1 – Le prix de vente doit être strictement po...
14,R2,Bloquante,ERP,4275,R2 – La quantité en stock ne doit jamais être ...
17,Structure,Bloquante,ERP,4713,Structure – stock_status : contrôle « isin(['i...
19,R5,Bloquante,Liaison,5764,R5 – L'identifiant web doit être numérique
821,R8,Bloquante,Web,14509,R8 – Le produit apparaît plusieurs fois dans l...
822,R8,Bloquante,Web,14509,R8 – Le produit apparaît plusieurs fois dans l...


,Règle,Table,Identifiant,Modification,Détectée
0,R1,ERP,6221,Prix de vente mis à 0,True
1,R2,ERP,4275,Stock mis à -5,True
2,R3,ERP,6585,"Stock mis à 0, statut instock conservé",True
3,R4,ERP,4632,Prix de vente égal à la moitié du prix d'achat,True
4,R5,Liaison,5764,Lettre ajoutée à l'identifiant web,True
5,R6,Liaison,4920,Identifiant web supprimé pour un produit en ve...,True
6,R8,Web,14509,Ligne produit dupliquée,True
7,R9,Web,15481,Ventes mises à -3,True
8,R10,Web,colonne_test_vide,Colonne entièrement vide ajoutée,True
9,R11,Web,15856,Balise HTML ajoutée au titre,True


Taux de détection : 100%


In [14]:
from openpyxl.styles import Font, Alignment

# Catalogue des règles (tâche 2), avec l'outil de correction et le responsable pressenti
CATALOGUE = pd.DataFrame([
    ("R1", "Le prix de vente est strictement positif", "ERP", "Équipe ventes"),
    ("R2", "La quantité en stock n'est jamais négative", "ERP", "Équipe ventes"),
    ("R3", "Le statut de stock correspond à la quantité", "ERP", "Équipe ventes"),
    ("R4", "Le prix de vente est supérieur ou égal au prix d'achat, sauf déstockage validé", "ERP", "Équipe ventes, validation de Nicolas"),
    ("R5", "L'identifiant web est toujours numérique", "Table de liaison", "Stagiaire en charge de la liaison"),
    ("R6", "Un produit en vente sur le site a un identifiant web", "Table de liaison", "Stagiaire en charge de la liaison"),
    ("R7", "Un produit retiré de la vente web n'a plus d'identifiant web actif", "Table de liaison", "Stagiaire en charge de la liaison"),
    ("R8", "L'export web ne contient que des produits, chacun une seule fois", "Paramétrage de l'export du site", "Responsable du site e-commerce"),
    ("R9", "Chaque produit du site a une référence et des ventes positives ou nulles", "Site (WordPress)", "Responsable du site e-commerce"),
    ("R10", "L'export ne contient pas de colonnes vides ou inutilisées", "Paramétrage de l'export du site", "Responsable du site e-commerce"),
    ("R11", "Les titres et descriptions ne contiennent pas de code HTML", "Site (WordPress)", "Responsable du site e-commerce"),
    ("R12", "Toute référence de la table de liaison existe sur le site", "Table de liaison", "Stagiaire en charge de la liaison"),
    ("R13", "Tout produit vendu sur le site existe dans l'ERP", "ERP et table de liaison", "Équipe ventes et stagiaire"),
    ("R14", "Les produits qui ne sont pas des vins ont un type dédié", "Site (WordPress)", "Responsable du site e-commerce"),
    ("R15", "Chaque produit de l'ERP a un libellé disponible, dans l'ERP ou sur le site", "ERP", "Équipe ventes"),
    ("Structure", "Les valeurs respectent le format attendu (types, valeurs autorisées)", "Outil d'origine de la donnée", "À déterminer selon la table"),
], columns=["Règle", "Énoncé métier", "Où corriger", "Responsable pressenti"])


def nom_produit(rapport, erp, web, liaison):
    """Ajoute le nom du produit à chaque anomalie, quand il est connu."""
    produits = web[web["post_type"] == "product"].dropna(subset=["sku"])
    nom_par_sku = dict(zip(produits["sku"].astype(str), produits["post_title"]))
    sku_par_pid = dict(zip(liaison["product_id"].astype(str), liaison["id_web"].astype("string")))
    def trouver(ident):
        if ident in nom_par_sku:
            return nom_par_sku[ident]
        sku = sku_par_pid.get(ident)
        return nom_par_sku.get(sku, "") if pd.notna(sku) else ""
    return rapport.assign(Produit=rapport["Identifiant"].map(trouver))


def exporter_rapport(rapport, erp, web, liaison, fichier="rapport_anomalies.xlsx"):
    """Produit le rapport Excel : un onglet Synthèse et un onglet Détail."""
    ordre = {"Bloquante": 0, "Avertissement": 1}

    detail = (nom_produit(rapport, erp, web, liaison)
              .merge(CATALOGUE[["Règle", "Où corriger"]], on="Règle", how="left")
              .assign(_o=lambda d: d["Gravité"].map(ordre))
              .sort_values(["_o", "Règle", "Identifiant"]).drop(columns="_o")
              [["Gravité", "Règle", "Table", "Identifiant", "Produit", "Message", "Où corriger"]])

    comptes = rapport.groupby("Règle").size().rename("Nombre d'anomalies")
    synthese = (CATALOGUE.assign(Gravité=CATALOGUE["Règle"].map(GRAVITE))
                .merge(comptes, left_on="Règle", right_index=True, how="left")
                .fillna({"Nombre d'anomalies": 0})
                .astype({"Nombre d'anomalies": int})
                .assign(_o=lambda d: d["Gravité"].map(ordre),
                        _n=lambda d: d["Règle"].str.extract(r"(\d+)")[0].astype(float))
                .sort_values(["_o", "_n"]).drop(columns=["_o", "_n"])
                [["Gravité", "Règle", "Énoncé métier", "Nombre d'anomalies", "Où corriger", "Responsable pressenti"]])

    with pd.ExcelWriter(fichier, engine="openpyxl") as writer:
        synthese.to_excel(writer, sheet_name="Synthèse", index=False)
        detail.to_excel(writer, sheet_name="Détail", index=False)
        for nom, df in [("Synthèse", synthese), ("Détail", detail)]:
            feuille = writer.sheets[nom]
            feuille.freeze_panes = "A2"                    # en-tête toujours visible
            feuille.auto_filter.ref = feuille.dimensions   # filtres sur chaque colonne
            for cellule in feuille[1]:
                cellule.font = Font(bold=True)
            for i, col in enumerate(df.columns, start=1):
                largeur = min(max(len(str(col)), df[col].astype(str).str.len().max()) + 2, 70)
                feuille.column_dimensions[feuille.cell(1, i).column_letter].width = largeur
                for colonne in feuille.iter_cols(min_col=i, max_col=i, min_row=2):
                    for c in colonne:
                        c.alignment = Alignment(wrap_text=True, vertical="top")
    return synthese, detail


synthese, detail = exporter_rapport(octobre, erp, web, liaison)
display(synthese)
print("Lignes dans l'onglet Détail :", len(detail))

,Gravité,Règle,Énoncé métier,Nombre d'anomalies,Où corriger,Responsable pressenti
0,Bloquante,R1,Le prix de vente est strictement positif,3,ERP,Équipe ventes
1,Bloquante,R2,La quantité en stock n'est jamais négative,2,ERP,Équipe ventes
3,Bloquante,R4,Le prix de vente est supérieur ou égal au prix...,4,ERP,"Équipe ventes, validation de Nicolas"
4,Bloquante,R5,L'identifiant web est toujours numérique,5,Table de liaison,Stagiaire en charge de la liaison
7,Bloquante,R8,"L'export web ne contient que des produits, cha...",797,Paramétrage de l'export du site,Responsable du site e-commerce
8,Bloquante,R9,Chaque produit du site a une référence et des ...,2,Site (WordPress),Responsable du site e-commerce
12,Bloquante,R13,Tout produit vendu sur le site existe dans l'ERP,0,ERP et table de liaison,Équipe ventes et stagiaire
15,Bloquante,Structure,Les valeurs respectent le format attendu (type...,0,Outil d'origine de la donnée,À déterminer selon la table
2,Avertissement,R3,Le statut de stock correspond à la quantité,2,ERP,Équipe ventes
5,Avertissement,R6,Un produit en vente sur le site a un identifia...,3,Table de liaison,Stagiaire en charge de la liaison


Lignes dans l'onglet Détail : 1170


In [15]:
MESSAGE_R8_LIGNES = "R8 – La ligne n'est pas un produit (image, ligne vide…)"

def regrouper_r8(rapport):
    """Remplace les lignes non-produits de R8 par une seule ligne récapitulative."""
    lignes_r8 = rapport["Message"] == MESSAGE_R8_LIGNES
    nb = int(lignes_r8.sum())
    if nb == 0:
        return rapport
    recap = pd.DataFrame([{"Règle": "R8", "Gravité": GRAVITE["R8"], "Table": "Web",
                           "Identifiant": f"{nb} lignes",
                           "Message": f"R8 – {nb} lignes d'images ou lignes vides dans l'export : "
                                      "une seule correction, dans le paramétrage de l'export"}])
    return pd.concat([rapport[~lignes_r8], recap], ignore_index=True)


def en_nombre(valeur):
    """Convertit un identifiant en nombre entier quand c'est possible, pour un tri correct dans Excel."""
    return int(valeur) if str(valeur).isdigit() else valeur


def exporter_rapport(rapport, erp, web, liaison, fichier="rapport_anomalies.xlsx"):
    """Produit le rapport Excel : un onglet Synthèse et un onglet Détail."""
    ordre = {"Bloquante": 0, "Avertissement": 1}

    detail = (nom_produit(regrouper_r8(rapport), erp, web, liaison)
              .merge(CATALOGUE[["Règle", "Où corriger"]], on="Règle", how="left")
              .assign(_o=lambda d: d["Gravité"].map(ordre))
              .sort_values(["_o", "Règle", "Identifiant"]).drop(columns="_o")
              [["Gravité", "Règle", "Table", "Identifiant", "Produit", "Message", "Où corriger"]])
    detail["Identifiant"] = detail["Identifiant"].map(en_nombre).astype(object)

    comptes = rapport.groupby("Règle").size().rename("Nombre d'anomalies")
    synthese = (CATALOGUE.assign(Gravité=CATALOGUE["Règle"].map(GRAVITE))
                .merge(comptes, left_on="Règle", right_index=True, how="left")
                .fillna({"Nombre d'anomalies": 0})
                .astype({"Nombre d'anomalies": int})
                .assign(_o=lambda d: d["Gravité"].map(ordre),
                        _n=lambda d: d["Règle"].str.extract(r"(\d+)")[0].astype(float))
                .sort_values(["_o", "_n"]).drop(columns=["_o", "_n"])
                [["Gravité", "Règle", "Énoncé métier", "Nombre d'anomalies", "Où corriger", "Responsable pressenti"]])

    with pd.ExcelWriter(fichier, engine="openpyxl") as writer:
        synthese.to_excel(writer, sheet_name="Synthèse", index=False)
        detail.to_excel(writer, sheet_name="Détail", index=False)
        for nom, df in [("Synthèse", synthese), ("Détail", detail)]:
            feuille = writer.sheets[nom]
            feuille.freeze_panes = "A2"
            feuille.auto_filter.ref = feuille.dimensions
            for cellule in feuille[1]:
                cellule.font = Font(bold=True)
            for i, col in enumerate(df.columns, start=1):
                largeur = min(max(len(str(col)), df[col].astype(str).str.len().max()) + 2, 70)
                feuille.column_dimensions[feuille.cell(1, i).column_letter].width = largeur
                for colonne in feuille.iter_cols(min_col=i, max_col=i, min_row=2):
                    for c in colonne:
                        c.alignment = Alignment(wrap_text=True, vertical="top")
    return synthese, detail


synthese, detail = exporter_rapport(octobre, erp, web, liaison)
print("Lignes dans l'onglet Détail :", len(detail))
display(detail[detail["Règle"] == "R8"])

Lignes dans l'onglet Détail : 374


,Gravité,Règle,Table,Identifiant,Produit,Message,Où corriger
373,Bloquante,R8,Web,797 lignes,,R8 – 797 lignes d'images ou lignes vides dans ...,Paramétrage de l'export du site


In [17]:
CA_P6 = 143324.10   # chiffre d'affaires présenté dans le P6

# Produits du site, avec la référence en texte
produits_site = web[web["post_type"] == "product"].dropna(subset=["sku"])
produits_site = produits_site.assign(sku=produits_site["sku"].astype(str))
liaison_txt = liaison.assign(id_web=liaison["id_web"].astype("string"))

# Table des ventes : produits du site reliés à l'ERP, sans suppression pour anomalie de stock ou de prix
ventes = (produits_site[["sku", "post_title", "total_sales"]]
          .merge(liaison_txt, left_on="sku", right_on="id_web", how="inner")
          .merge(erp, on="product_id", how="inner"))
ventes["CA"] = ventes["price"] * ventes["total_sales"]

ca_total = ventes["CA"].sum()
print("Produits vendus sur le site et reliés à l'ERP :", len(ventes))
print(f"CA web recalculé : {ca_total:,.2f} €")
print(f"CA web du P6     : {CA_P6:,.2f} €")
print(f"Écart            : {ca_total - CA_P6:,.2f} €")

# Produits présents dans le recalcul mais supprimés par le nettoyage du P6 (prix ou stock négatif)
supprimes_p6 = erp.loc[(erp["price"] < 0) | (erp["stock_quantity"] < 0), "product_id"]
display(ventes[ventes["product_id"].isin(supprimes_p6)][["product_id", "post_title", "price", "total_sales", "CA"]])

Produits vendus sur le site et reliés à l'ERP : 714
CA web recalculé : 143,680.10 €
CA web du P6     : 143,324.10 €
Écart            : 356.00 €


,product_id,post_title,price,total_sales,CA
536,5700,"Gilles Robin Crozes-Hermitage Rouge ""1920"" 2016",44.5,8.0,356.0


In [18]:
import numpy as np

PARETO_P6 = 44.1   # part du CA des 20 % meilleurs produits, dans le P6 (calculée sur 820 produits)

ca_tries = ventes["CA"].sort_values(ascending=False).reset_index(drop=True)
part_cumulee = ca_tries.cumsum() / ca_tries.sum()

# 1. Part du CA réalisée par les 20 % meilleurs produits du site
nb_top20 = int(np.ceil(0.20 * len(ca_tries)))
part_top20 = ca_tries.head(nb_top20).sum() / ca_tries.sum() * 100

# 2. Part des produits nécessaire pour atteindre 80 % du CA
nb_pour_80 = int((part_cumulee < 0.80).sum()) + 1
part_produits_80 = nb_pour_80 / len(ca_tries) * 100

print(f"Produits du site : {len(ca_tries)} (dont {int((ventes['CA'] == 0).sum())} sans vente en octobre)")
print(f"Les 20 % meilleurs produits ({nb_top20}) réalisent {part_top20:.1f} % du CA (P6 : {PARETO_P6} %)")
print(f"Il faut {nb_pour_80} produits, soit {part_produits_80:.1f} % du catalogue web, pour atteindre 80 % du CA")

Produits du site : 714 (dont 25 sans vente en octobre)
Les 20 % meilleurs produits (143) réalisent 40.2 % du CA (P6 : 44.1 %)
Il faut 435 produits, soit 60.9 % du catalogue web, pour atteindre 80 % du CA


In [19]:
STOCK_P6 = 298540.07          # valeur du stock au prix d'achat, dans le P6
MARGE_LATENTE_P6 = 233406.13  # marge latente, dans le P6

e = erp.copy()
e["stock_valide"] = e["stock_quantity"].clip(lower=0)            # stock négatif compté à 0
prix_valide = e["price"] > 0                                      # R1 respectée
vente_a_perte = prix_valide & (e["price"] < e["purchase_price"])  # R4

# 1. Valeur du stock au prix d'achat
valeur_stock = (e["stock_valide"] * e["purchase_price"]).sum()

# 2. Marge latente, avec et sans les ventes à perte
marge_latente = e["stock_valide"] * (e["price"] - e["purchase_price"])
ml_toutes = marge_latente[prix_valide].sum()
ml_sans_r4 = marge_latente[prix_valide & ~vente_a_perte].sum()

print(f"Valeur du stock : {valeur_stock:,.2f} € (P6 : {STOCK_P6:,.2f} €, écart : {valeur_stock - STOCK_P6:,.2f} €)")
print(f"Marge latente, tous produits à prix valide : {ml_toutes:,.2f} € (P6 : {MARGE_LATENTE_P6:,.2f} €)")
print(f"Marge latente, hors ventes à perte (R4)   : {ml_sans_r4:,.2f} €")
print(f"Effet des ventes à perte sur la marge     : {ml_toutes - ml_sans_r4:,.2f} €")

display(e[vente_a_perte][["product_id", "price", "purchase_price", "stock_valide"]]
        .assign(marge_latente=marge_latente[vente_a_perte]))

Valeur du stock : 298,627.66 € (P6 : 298,540.07 €, écart : 87.59 €)
Marge latente, tous produits à prix valide : 233,406.13 € (P6 : 233,406.13 €)
Marge latente, hors ventes à perte (R4)   : 239,831.64 €
Effet des ventes à perte sur la marge     : -6,425.51 €


,product_id,price,purchase_price,stock_valide,marge_latente
210,4355,12.65,77.48,97,-6288.51
391,4864,8.30,9.99,0,-0.00
724,6324,92.00,99.00,18,-126.00
817,7196,31.00,31.20,55,-11.00
